# 01. Treinar Splink 5 no Spark — UF21 (mãe + endereço)

EM no Spark com `censo_limpo` / `cpf_limpo` (UF21).
Comparisons iguais ao 02c DuckDB (nome, data, mãe, endereço; sem UF).

Entrada: `censo_limpo.parquet`, `cpf_limpo.parquet` em
`/data/spark/shared/datasets/`.

Saída: `splink_model_mae_endereco.json` (mesmo diretório).

Apply: [`02_aplicar_mae_endereco_uf21.ipynb`](02_aplicar_mae_endereco_uf21.ipynb).


In [ ]:
import os
from pathlib import Path

import json

import splink
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType
from splink.backends.spark import SparkAPI

# 'small' ou 'full' — ver spark_cluster.json
CLUSTER = 'small'

_cfg_candidates = [
    Path.cwd() / 'spark_cluster.json',
    Path.cwd() / 'spark' / 'spark_cluster.json',
    Path.cwd() / 'notebooks' / 'spark' / 'spark_cluster.json',
]
CFG_PATH = next(p for p in _cfg_candidates if p.exists())
cfg_all = json.loads(CFG_PATH.read_text())
profile = cfg_all['profiles'][CLUSTER]

os.environ['JAVA_HOME'] = cfg_all['java_home']
os.environ['SPARK_HOME'] = cfg_all['spark_home']
os.environ['PYSPARK_PYTHON'] = cfg_all['python']
os.environ['PYSPARK_DRIVER_PYTHON'] = cfg_all['python']

jar_path = cfg_all['jar']
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    f'--jars {jar_path} '
    f'--driver-class-path {jar_path} '
    'pyspark-shell'
)

builder = (
    SparkSession.builder
    .appName('01_treinar_mae_endereco_uf21')
    .master(cfg_all['master'])
    .config('spark.driver.host', cfg_all['driver']['host'])
    .config('spark.driver.bindAddress', cfg_all['driver']['bindAddress'])
    .config('spark.driver.port', cfg_all['driver']['port'])
    .config('spark.pyspark.python', cfg_all['python'])
)
for key, value in profile.items():
    builder = builder.config(key, value)

spark = builder.getOrCreate()

DATA_DIR = Path('/data/spark/shared/datasets')
CKPT_DIR = cfg_all['checkpoint_dir']
spark.sparkContext.setCheckpointDir(CKPT_DIR)
spark.sparkContext.setLogLevel('ERROR')
ckpt = spark.sparkContext.getCheckpointDir()
if not ckpt:
    raise RuntimeError('Checkpoint dir nao configurado no SparkContext')

print('Spark:', spark.version)
print('Master:', spark.sparkContext.master)
print('App:', spark.sparkContext.applicationId)
print('cluster profile:', CLUSTER)
print('splink:', getattr(splink, '__version__', '?'))
print('jar:', jar_path)
print('DATA_DIR:', DATA_DIR)
print('checkpoint:', ckpt)

CENSO_PARQUET = DATA_DIR / 'censo_limpo_uf21.parquet'
CPF_PARQUET = DATA_DIR / 'cpf_limpo_uf21.parquet'
MODEL_JSON = DATA_DIR / 'splink_model_mae_endereco.json'

KEEP_COLS = [
    'unique_id',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
    'cpf_norm',
]

print('modelo (saida):', MODEL_JSON)
print('censo:', CENSO_PARQUET)
print('cpf:  ', CPF_PARQUET)


In [ ]:
df_censo = (
    spark.read.parquet(str(CENSO_PARQUET))
    .select(*KEEP_COLS)
    .withColumn('idade', F.col('idade').cast(IntegerType()))
    .withColumn('source_dataset', F.lit('censo'))
)
df_cpf = (
    spark.read.parquet(str(CPF_PARQUET))
    .select(*KEEP_COLS)
    .withColumn('idade', F.col('idade').cast(IntegerType()))
    .withColumn('source_dataset', F.lit('cpf'))
)

print('censo:', df_censo.count())
print('cpf:  ', df_cpf.count())
print(
    'censo com cpf_norm:',
    df_censo.filter(F.col('cpf_norm').isNotNull()).count(),
)


## Comparisons


In [ ]:
import splink.comparison_level_library as cll
import splink.comparison_library as cl

JW_MAE_ALTO = 0.92
JW_MAE_MEDIO = 0.85
JW_LOGR_SIMILAR = 0.90

null_sql = (
    '(data_nascimento_l IS NULL OR data_nascimento_r IS NULL) AND '
    '(idade_l IS NULL OR idade_r IS NULL) AND '
    '(mes_nascimento_l IS NULL OR mes_nascimento_r IS NULL OR '
    'dia_nascimento_l IS NULL OR dia_nascimento_r IS NULL)'
)
dob_presente_sql = (
    'data_nascimento_l IS NOT NULL AND data_nascimento_r IS NOT NULL'
)
troca_mes_dia_sql = (
    'ano_nascimento_l = ano_nascimento_r AND '
    'mes_nascimento_l = dia_nascimento_r AND '
    'dia_nascimento_l = mes_nascimento_r AND '
    'mes_nascimento_l <> dia_nascimento_l'
)
dia_diferente_sql = (
    'ano_nascimento_l = ano_nascimento_r AND '
    'mes_nascimento_l = mes_nascimento_r AND '
    'dia_nascimento_l <> dia_nascimento_r'
)
mes_diferente_sql = (
    'ano_nascimento_l = ano_nascimento_r AND '
    'dia_nascimento_l = dia_nascimento_r AND '
    'mes_nascimento_l <> mes_nascimento_r'
)
ano_pm1_sql = (
    'mes_nascimento_l = mes_nascimento_r AND '
    'dia_nascimento_l = dia_nascimento_r AND '
    'abs(try_cast(ano_nascimento_l AS INTEGER) - '
    'try_cast(ano_nascimento_r AS INTEGER)) = 1'
)

# SQL Spark (não DuckDB): split/size/slice/element_at/jaro_winkler
split_l = "split(nome_completo_phon_l, ' ')"
split_r = "split(nome_completo_phon_r, ' ')"
n_min = f'least(size({split_l}), size({split_r}))'
prefixo_sql = (
    f'{n_min} >= 2 AND size({split_l}) <> size({split_r}) AND '
    f'slice({split_l}, 1, {n_min}) = slice({split_r}, 1, {n_min})'
)
jw_ultimo_095_sql = (
    f'element_at({split_l}, -1) = element_at({split_r}, -1) AND '
    'jaro_winkler(nome_completo_phon_l, nome_completo_phon_r) >= 0.95'
)
jw_ultimo_092_sql = (
    f'element_at({split_l}, -1) = element_at({split_r}, -1) AND '
    'jaro_winkler(nome_completo_phon_l, nome_completo_phon_r) >= 0.92'
)
dl_completo_1_sql = (
    'damerau_levenshtein(nome_completo_phon_l, nome_completo_phon_r) <= 1'
)
n_tok = f'size({split_l})'
mesmo_n = f'{n_tok} = size({split_r}) AND {n_tok} >= 3'
idx = f'sequence(1, size({split_l}))'
dif_count = (
    f'aggregate({idx}, CAST(0 AS INT), '
    f'(acc, i) -> acc + CASE WHEN element_at({split_l}, i) <> element_at({split_r}, i) '
    f'THEN 1 ELSE 0 END)'
)
dif_max_dl = (
    f'aggregate({idx}, CAST(0 AS INT), '
    f'(acc, i) -> greatest(acc, CASE WHEN element_at({split_l}, i) <> element_at({split_r}, i) '
    f'THEN CAST(damerau_levenshtein(element_at({split_l}, i), element_at({split_r}, i)) AS INT) '
    f'ELSE 0 END))'
)
um_token_sql = (
    f'{mesmo_n} AND element_at({split_l}, -1) = element_at({split_r}, -1) '
    f'AND {dif_count} = 1 AND {dif_max_dl} <= 2'
)
dois_tokens_sql = (
    f'{mesmo_n} AND {dif_count} BETWEEN 1 AND 2 AND {dif_max_dl} <= 2'
)
n_l = f'size({split_l})'
n_r = f'size({split_r})'
ordem_sql = (
    f'{n_l} = {n_r} AND {n_l} >= 3 '
    f'AND array_sort({split_l}) = array_sort({split_r}) '
    f'AND {split_l} <> {split_r}'
)

def _sem_token(split, k):
    n = f'size({split})'
    if k == 1:
        return f'slice({split}, 2, {n} - 1)'
    return (
        f'concat(slice({split}, 1, {k - 1}), '
        f'slice({split}, {k + 1}, {n} - {k}))'
    )

_ors_token = []
for _k in range(1, 11):
    _ors_token.append(f'({n_l} >= {_k} AND {_sem_token(split_l, _k)} = {split_r})')
    _ors_token.append(f'({n_r} >= {_k} AND {_sem_token(split_r, _k)} = {split_l})')
token_a_mais_sql = (
    f'abs({n_l} - {n_r}) = 1 AND least({n_l}, {n_r}) >= 3 AND ('
    + ' OR '.join(_ors_token)
    + ')'
)

# --- mãe ---
split_mae_l = "split(nome_mae_phon_l, ' ')"
split_mae_r = "split(nome_mae_phon_r, ' ')"
n_mae = f'least(size({split_mae_l}), size({split_mae_r}))'
mae_contem_sql = (
    f'{n_mae} >= 2 AND '
    f'size({split_mae_l}) <> size({split_mae_r}) AND '
    f'size(array_intersect({split_mae_l}, {split_mae_r})) = {n_mae}'
)
mae_ordem_sql = (
    f'size({split_mae_l}) = size({split_mae_r}) AND size({split_mae_l}) >= 3 '
    f'AND array_sort({split_mae_l}) = array_sort({split_mae_r}) '
    f'AND nome_mae_phon_l <> nome_mae_phon_r'
)
mae_jw_alto_sql = (
    f'jaro_winkler(nome_mae_phon_l, nome_mae_phon_r) >= {JW_MAE_ALTO}'
)
mae_jw_medio_sql = (
    f'jaro_winkler(nome_mae_phon_l, nome_mae_phon_r) >= {JW_MAE_MEDIO}'
)

# --- endereço composto ---
endereco_null_sql = (
    '(logradouro_norm_l IS NULL OR logradouro_norm_r IS NULL) AND '
    '(cep_l IS NULL OR cep_r IS NULL) AND '
    '(cod_municipio_l IS NULL OR cod_municipio_r IS NULL)'
)
logr_cep_exact_sql = (
    'logradouro_norm_l = logradouro_norm_r AND cep_l = cep_r'
)
logr_mun_exact_sql = (
    'logradouro_norm_l = logradouro_norm_r AND '
    'cod_municipio_l = cod_municipio_r'
)
logr_sim_cep_sql = (
    f'jaro_winkler(logradouro_norm_l, logradouro_norm_r) '
    f'>= {JW_LOGR_SIMILAR} AND cep_l = cep_r'
)
cep_exact_sql = 'cep_l = cep_r'

comparisons = [
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('nome_completo_phon'),
            cll.ExactMatchLevel(
                'nome_completo_phon', term_frequency_adjustments=True
            ),
            cll.CustomLevel(prefixo_sql, label_for_charts='Prefixo de tokens'),
            cll.CustomLevel(dl_completo_1_sql, label_for_charts='DL <= 1'),
            cll.CustomLevel(um_token_sql, label_for_charts='Um token DL <= 2'),
            cll.CustomLevel(
                dois_tokens_sql, label_for_charts='Ate dois tokens DL <= 2'
            ),
            cll.CustomLevel(
                token_a_mais_sql, label_for_charts='Um token a mais'
            ),
            cll.CustomLevel(
                ordem_sql, label_for_charts='Mesmos tokens, ordem diferente'
            ),
            cll.CustomLevel(
                jw_ultimo_095_sql, label_for_charts='Ultimo token e JW >= 0.95'
            ),
            cll.CustomLevel(
                jw_ultimo_092_sql, label_for_charts='Ultimo token e JW >= 0.92'
            ),
            cll.ElseLevel(),
        ],
        output_column_name='nome_completo_phon',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.CustomLevel(null_sql, label_for_charts='Null data e mes/dia').configure(
                is_null_level=True
            ),
            cll.ExactMatchLevel('data_nascimento', term_frequency_adjustments=True),
            cll.CustomLevel(
                troca_mes_dia_sql, label_for_charts='Mes e dia trocados'
            ),
            cll.CustomLevel(
                dia_diferente_sql, label_for_charts='Dia diferente'
            ),
            cll.CustomLevel(
                mes_diferente_sql, label_for_charts='Mes diferente'
            ),
            cll.CustomLevel(ano_pm1_sql, label_for_charts='Ano +/- 1'),
            cll.DamerauLevenshteinLevel('data_nascimento', 1),
            cll.DamerauLevenshteinLevel('data_nascimento', 2),
            cll.CustomLevel(
                dob_presente_sql,
                label_for_charts='Datas presentes, demais diferenças',
            ),
            cll.ExactMatchLevel('idade'),
            cll.AbsoluteDifferenceLevel('idade', 1),
            cll.ElseLevel(),
        ],
        output_column_name='data_nascimento',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('nome_mae_phon'),
            cll.ExactMatchLevel(
                'nome_mae_phon', term_frequency_adjustments=True
            ),
            cll.CustomLevel(mae_contem_sql, label_for_charts='Contida'),
            cll.CustomLevel(
                mae_ordem_sql, label_for_charts='Mesmos tokens, ordem diferente'
            ),
            cll.CustomLevel(
                mae_jw_alto_sql, label_for_charts=f'JW >= {JW_MAE_ALTO}'
            ),
            cll.CustomLevel(
                mae_jw_medio_sql, label_for_charts=f'JW >= {JW_MAE_MEDIO}'
            ),
            cll.ElseLevel(),
        ],
        output_column_name='nome_mae_phon',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.CustomLevel(
                endereco_null_sql, label_for_charts='Null / nao comparavel'
            ).configure(is_null_level=True),
            cll.CustomLevel(
                logr_cep_exact_sql, label_for_charts='Logradouro e CEP exact'
            ),
            cll.CustomLevel(
                logr_mun_exact_sql,
                label_for_charts='Logradouro e municipio exact',
            ),
            cll.CustomLevel(
                logr_sim_cep_sql,
                label_for_charts='Logradouro similar e CEP exact',
            ),
            cll.CustomLevel(cep_exact_sql, label_for_charts='CEP exact'),
            cll.ElseLevel(),
        ],
        output_column_name='endereco',
    ),
]


## Linker


In [ ]:
from splink import Linker, SettingsCreator

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    retain_intermediate_calculation_columns=False,
)

# Splink 5: register + Linker([SplinkDataFrame, ...], settings)
spark.sparkContext.setCheckpointDir(CKPT_DIR)
db_api = SparkAPI(
    spark_session=spark,
    break_lineage_method='checkpoint',
    repartition_after_blocking=False,
)
censo_in = db_api.register(df_censo, dataset_display_name='censo')
cpf_in = db_api.register(df_cpf, dataset_display_name='cpf')
linker = Linker([censo_in, cpf_in], settings)

print('Linker treino pronto')
print('censo:', df_censo.count(), '| cpf:', df_cpf.count())


## Prior e u


In [ ]:
from splink import block_on

U_MAX_PAIRS = 500_000_000
EM_MAX_PAIRS = 10_000_000

deterministic_rules = [
    block_on('cpf_norm'),
    block_on('nome_completo_phon', 'data_nascimento'),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.7
)
print('max_pairs u:', f'{U_MAX_PAIRS:,}')
linker.training.estimate_u_using_random_sampling(max_pairs=U_MAX_PAIRS)


## EM


In [ ]:
# m direto: mesmo cpf_norm é match. NULL não casa com NULL.
linker.training.estimate_m_from_label_column('cpf_norm')

# Nome. A data está na chave, então a comparison temporal sai desta sessão.
em_nome = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('data_nascimento', 'sexo'),
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)

# Data. Primeiro+último não desliga nome_completo; a condição do nome
# é sempre verdadeira e só tira essa comparison da sessão.
bloqueio_data = (
    'l.primeiro_nome_phon = r.primeiro_nome_phon AND '
    'l.ultimo_nome_phon = r.ultimo_nome_phon AND '
    '(l.nome_completo_phon IS NULL OR l.nome_completo_phon IS NOT NULL)'
)
em_data = linker.training.estimate_parameters_using_expectation_maximisation(
    bloqueio_data,
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)
m_data = linker._settings_obj.core_model_settings.copy()

# Diagnóstico: o mesmo bloco com sexo. Não fica no modelo salvo.
bloqueio_data_sexo = bloqueio_data + ' AND l.sexo = r.sexo'
em_data_sexo = linker.training.estimate_parameters_using_expectation_maximisation(
    bloqueio_data_sexo,
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)
linker._settings_obj.core_model_settings = m_data
linker._populate_m_u_from_trained_values()

# Mãe: nome+data na chave → comparison de mãe treina.
em_mae = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('nome_completo_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)

# Endereço: primeiro+último+data (endereço fora da chave).
em_endereco = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
    fix_u_probabilities=True,
    max_pairs=EM_MAX_PAIRS,
)



## Salvar modelo


In [ ]:
MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(MODEL_JSON), overwrite=True)
print('Modelo salvo:', MODEL_JSON)
